# What Is an AI Agent?

## What you'll learn
- The difference between a single model call, a fixed workflow, and an agent
- The three parts of every agent: a model, tools, and a loop
- The FP&A dataset used throughout this stage
- How to answer a variance question by hand with pandas (the job the agent will do later)

This lesson needs no API key.

## From one call to an agent

A **large language model (LLM)** such as Claude or ChatGPT takes text in and gives text
back. There are three common ways to use one in code:

| Approach | What happens | Example |
|----------|--------------|---------|
| Single call | You send one prompt, you get one answer | "Summarise this paragraph" |
| Workflow | Your code runs fixed steps, some of them model calls | Load data, ask for commentary, save a report |
| Agent | The model decides which steps to take, calling your tools until it is done | "Why did Q2 operating income miss budget?" |

An **agent** is a model plus **tools** (Python functions it may ask you to run) plus a
**loop** that keeps going until the model has an answer:

1. You send the question and a list of tools.
2. The model replies either with an answer, or with a request: "run this tool with these inputs".
3. Your code runs the tool and sends the result back.
4. Repeat from step 2.

The model never touches your data directly. It only sees what your tools return, which is
what makes agents safe to build on business data.

## The dataset

`data/fpa_actuals_budget.csv` holds three years of monthly actuals and budget for a
fictional company, by product line and region. Every amount is in USD thousands.
We also add the derived P&L lines an FP&A team always looks at: gross profit and
operating income.

In [ ]:
import json
import pandas as pd

df = pd.read_csv("data/fpa_actuals_budget.csv")

# Derived lines every FP&A pack uses (all amounts in USD thousands).
df["Gross_Profit"] = df["Revenue"] - df["COGS"]
df["Budget_Gross_Profit"] = df["Budget_Revenue"] - df["Budget_COGS"]
df["Operating_Income"] = df["Gross_Profit"] - df["OpEx"]
df["Budget_Operating_Income"] = df["Budget_Gross_Profit"] - df["Budget_OpEx"]

print(df.shape)
df.head()

In [ ]:
print("Months:", df["Month"].min(), "to", df["Month"].max())
print("Product lines:", sorted(df["ProductLine"].unique()))
print("Regions:", sorted(df["Region"].unique()))

## Variance: the core FP&A question

**Variance** is actual minus budget. For revenue and profit, a positive variance is good
(favourable). For costs such as COGS and OpEx, a positive variance means you spent more
than planned, which is unfavourable.

In [ ]:
q2 = df[(df["Month"] >= "2025-04") & (df["Month"] <= "2025-06")]

lines = ["Revenue", "COGS", "Gross_Profit", "OpEx", "Operating_Income"]
summary = pd.DataFrame({
    "actual": [q2[m].sum() for m in lines],
    "budget": [q2[f"Budget_{m}"].sum() for m in lines],
}, index=lines)
summary["variance"] = summary["actual"] - summary["budget"]
summary["variance_pct"] = summary["variance"] / summary["budget"] * 100
summary.round(1)

## Finding the driver

A total variance is only the start. The analyst's next question is "where did it come
from?", which means splitting the variance by product line or region.

In [ ]:
by_region = q2.groupby("Region")[["Revenue", "Budget_Revenue"]].sum()
by_region["variance"] = by_region["Revenue"] - by_region["Budget_Revenue"]
by_region.sort_values("variance").round(1)

Europe stands out. An analyst would now look at Europe month by month, check the FX
rate, and split by product line. Each of those is one small, well-defined question, which
is exactly what we will turn into **tools** for the agent in lesson 4.

## Practice

1. Build the Q2 2025 (`2025-04` to `2025-06`) variance table for **Operating_Income** only,
   split by product line. Which product line missed budget by the most?
2. For **Europe** in Q2 2025, show revenue actual, budget, and the average `FX_Rate` by month.
3. Compute the **gross margin %** (gross profit divided by revenue, times 100) for
   Diagnostics in North America, actual vs budget, for Q2 2025.
4. In one sentence each, would you use a single call, a workflow, or an agent for:
   (a) translating a paragraph, (b) a monthly report with the same steps every time,
   (c) answering open questions about the P&L?

In [ ]:
# Your practice code here

## Summary

- A single call answers one prompt; a workflow runs fixed steps; an agent chooses its own steps.
- An agent is a model, a set of tools, and a loop.
- The model only sees what your tools return.
- Variance is actual minus budget; for costs, a positive variance is unfavourable.
- Good tools answer small, well-defined questions, like the pandas steps above.

## Practice Solutions

In [ ]:
import json
import pandas as pd

df = pd.read_csv("data/fpa_actuals_budget.csv")

# Derived lines every FP&A pack uses (all amounts in USD thousands).
df["Gross_Profit"] = df["Revenue"] - df["COGS"]
df["Budget_Gross_Profit"] = df["Budget_Revenue"] - df["Budget_COGS"]
df["Operating_Income"] = df["Gross_Profit"] - df["OpEx"]
df["Budget_Operating_Income"] = df["Budget_Gross_Profit"] - df["Budget_OpEx"]



q2 = df[(df["Month"] >= "2025-04") & (df["Month"] <= "2025-06")]

# 1. Operating income variance by product line
oi = q2.groupby("ProductLine")[["Operating_Income", "Budget_Operating_Income"]].sum()
oi["variance"] = oi["Operating_Income"] - oi["Budget_Operating_Income"]
print(oi.sort_values("variance").round(1), "\n")

# 2. Europe by month with FX
europe = q2[q2["Region"] == "Europe"].groupby("Month").agg(
    revenue=("Revenue", "sum"), budget=("Budget_Revenue", "sum"), fx_rate=("FX_Rate", "mean"))
print(europe.round(3), "\n")

# 3. Gross margin % for Diagnostics, North America
dx = q2[(q2["ProductLine"] == "Diagnostics") & (q2["Region"] == "North America")]
actual_gm = dx["Gross_Profit"].sum() / dx["Revenue"].sum() * 100
budget_gm = dx["Budget_Gross_Profit"].sum() / dx["Budget_Revenue"].sum() * 100
print(f"Gross margin: actual {actual_gm:.1f}% vs budget {budget_gm:.1f}%")

# 4. (a) single call, (b) workflow, (c) agent